# Assignment-12: RAG - Retrieval Augmented Generation

### Install Dependencies

In [6]:
# %%capture
# !pip -q install langchain==0.3.27
# !pip -q install openai==2.3.0
# !pip -q install langchain-core==0.3.79
# !pip -q install langchain-community==0.3.31
# !pip -q install sentence-transformers==5.1.1
# !pip -q install langchain-huggingface==0.3.1
# !pip -q install langchain-experimental==0.3.4
# !pip -q install langchainhub==0.1.21
# !pip -q install langchain-openai==0.3.35
# !pip -q install langchain-chroma==0.2.6
# !pip -q install chromadb==1.1.1
# !pip -q install pypdf==6.1.1

In [7]:
# !pip install tf-keras

### Import Required Packages

In [8]:
import os
from google import genai  # The official modern Google GenAI library
import numpy as np
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Modern Google LangChain integrations
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Note: LangChain will automatically look for os.environ["GEMINI_API_KEY"] or os.environ["GOOGLE_API_KEY"]

# 1. Initialize the Google Embedding Model for Chroma
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

# 2. Initialize the Google Chat Model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0.7
)

# --- Your RAG pipeline assembly can continue below ---
# Example Chroma setup:
# vectorstore = Chroma(embedding_function=embeddings, persist_directory="./chroma_db")


### Provide your OpenAI API Key - Google API Key

In [9]:
import os
from dotenv import load_dotenv

load_dotenv()
# The SDKs will automatically read os.environ["GEMINI_API_KEY"]

True

In [10]:
from google import genai

client = genai.Client()

# Fix: Change text-embedding-004 to gemini-embedding-001
embedding_response = client.models.embed_content(
    model="gemini-embedding-001",
    contents="Your chunked document text goes here"
)

# Access your embeddings
print(embedding_response.embeddings)


[ContentEmbedding(
  values=[
    0.0017330004,
    0.0060653132,
    0.01663676,
    -0.05727272,
    0.008927235,
    <... 3067 more items ...>,
  ]
)]


In [11]:
import os
from google import genai
from google.genai.errors import APIError

try:
    client = genai.Client()
    
    # Updated to the current standard model
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents="Respond with exactly the words: API connection successful!"
    )
    
    print("✅ Success! Your Google API key is configured correctly.")
    print(f"Model Response: {response.text.strip()}")

except APIError as e:
    print("❌ Connection Failed! The API returned an error:")
    print(e)
except Exception as e:
    print("❌ An unexpected error occurred.")
    print(e)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


✅ Success! Your Google API key is configured correctly.
Model Response: API connection successful!


### Load LLM

In [12]:
# pip install -qU langchain-google-genai

In [13]:
# Fix the import name here:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize the Gemini Model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0.7,
)

# Test the connection
response = llm.invoke("Hello! Confirm you are working within LangChain.")
print(response.content)


[{'type': 'text', 'text': 'Hello! I cannot confirm whether I am currently running within LangChain. \n\nAs an AI model, I only receive the text of your prompt and do not have visibility into the backend architecture, client code, or orchestration frameworks (like LangChain, raw API calls, LlamaIndex, etc.) being used to send and receive these messages. \n\nIf you are developing an application with LangChain, however, I can certainly help you write, debug, or understand LangChain code!', 'extras': {'signature': 'EqsLCqgLAWkUfRNysy4m0mAVIJiFOxAQJAFyYjK7REDuHNZ4o5XIY2EDWCbjzKKi83fq/W0qdSATi1Z0/SRVQyAr36fYYrIec7Va4DvmRcu+nTeC3sCWvncI7CuikuoWkObsbWe+Y9y4ireJtRuPox4kwZ4KCSXUPzw5YdYjMF+0Gxqaf1H62pLskwddZFjH/WaHwva7wpmisTNM9uGYSH5OOQwKknohPzJUVKk3xCCm7jnS6vLxXuOoqZ7YlJseXd/s+iQM0Ot/mvoVQOCZ80ofix2nb/bLHXPV3D2G7AkG3JyEY4WehLd35Ou5kOZpOVH3MxjiM4i4dseW3Pu9xeeGJx0kYMPJLAmDmHD3P72vdBeHo+W8uvH5atneMj6FoWA0Lp9ERsrLIG58qoIfjW7vpxYJFnnec41sQ6aX9YVZQMvFKZVGdtTuM16WLWtUFY5Xpt3bK+5UnLiilQdlmMoDuBi4y9Ba9OO

### General Query

In [14]:
# General query
response = llm.invoke("What is the Capital of India?")
print(response.content)

[{'type': 'text', 'text': 'The capital of India is **New Delhi**.', 'extras': {'signature': 'Et0CCtoCAWkUfRPVbtscnJcxVxtw60Z0Y4oX5xI2toSlWZe7arlgJpxcOY2NpQzx6IC/McNSR520VtyOsRN435W5aD5tOvO2bxOxWcfkaf7LEsBnyhs3rumavZHcxdIc56LKV3NfL+BM63R6bvFqTWDgPXyHgG5S/E3iwSOc3GsNR3EM+EbRpBDSt4/+AitlEkEyjizra21f+QLnM71gIGcCNjqX+2oRZRKq8x3eHzv3GMFfUm4o3LQnmbzVCKbyaKk/F+d+JrNfEZTB9ixjcAdtUq/DLF2k20KTJUteqBMhSlUo6PMUXt4B++mT/GeGhISyezK9lfJuhREoaMudTXT6ulaAqTrdPGn6ff03chnz7OsCmq/FLwOuvOtSUyZCZzi0wvJY1MYgv19ZSrE/ofUuqxF5sob6WI0M3pfbj/l/s4WRMICiHwBq5xaNwyUFZBQ66oeG/8DxoxLPTZ2Sbg=='}}]


## Steps: 

1. **Document Loading:** Ingest the raw data files using loaders like `PyPDFLoader` to extract the text content into the pipeline.


2. **Text Splitting (Chunking):** Break the extracted text into smaller, manageable pieces using `RecursiveCharacterTextSplitter`. This ensures the text fits within the LLM's context window while maintaining semantic continuity.


3. **Embedding and Vector Storage:** Convert the text chunks into dense vector representations using an embedding model (such as `OpenAIEmbeddings`) and store them in a persistent vector database like `Chroma` for efficient similarity search.


4. **Retrieval:** Fetch the most relevant text chunks from the vector database based on the user's query. This step must handle specific edge cases:


* **Addressing Diversity:** Using Maximum Marginal Relevance (MMR) to avoid fetching duplicate or highly redundant chunks.


* **Addressing Specificity:** Using metadata filtering (either manually or via a self-query retriever) to ensure answers are drawn from the correct source document.




5. **Augmentation and Generation:** Construct a LangChain Expression Language (LCEL) pipeline using `RunnableParallel` and `RunnablePassthrough` to pass the user's question and the retrieved context into a formatting `PromptTemplate`. The LLM then reads this augmented prompt to generate a final, grounded answer.

### Step-1: Document Loading

Description: In this initial step, we ingest the raw data files to extract their text content into our pipeline. We use PyPDFLoader to load the research paper, converting each page of the PDF into a Document object that contains both the text (page_content) and metadata (such as the source file and page number).

In [15]:
import os
os.path.abspath(os.getcwd()) # shows current

'c:\\Users\\Vishwas Joshi\\OBELIX\\iitm-applied-ml-portfolio\\04_generative_ai\\rag\\src'

In [16]:
os.chdir('C:/Users/Vishwas Joshi/OBELIX/iitm-applied-ml-portfolio/04_generative_ai/rag') # changes it

In [17]:
from langchain_community.document_loaders import PyPDFLoader

# Initialize the loader with the specific research paper path
loaders = [
    PyPDFLoader("data/Evaluation_of_the_Robustness_of_Machine_Learning.pdf")
]

# Extract the content from the document
docs = []
for loader in loaders:
    docs.extend(loader.load())

print(f"Successfully loaded {len(docs)} pages.")

Successfully loaded 12 pages.


In [18]:
len(docs)

12

In [19]:
# print(docs[0].page_content)

### Step 2: Text Splitting (Chunking)

**Description:** Breaking the extracted text into smaller chunks using `RecursiveCharacterTextSplitter`. This step ensures the text fits within the LLM's context window while maintaining semantic continuity by preserving paragraphs and overlapping boundaries.

**Overview & Strategy:**
In a RAG architecture, retrieval accuracy depends on embedding focused passages rather than full documents. Embedding full pages creates diffuse vectors where specific facts are lost, while also driving up LLM token usage and latency during generation.

We use LangChain's `RecursiveCharacterTextSplitter` to partition the loaded text into structured, overlapping chunks.

**Key Mechanics:**
* **Hierarchical Splitting:** Tries splitting in order of semantic preservation: paragraphs (`\n\n`), sentences (`\n`), words (` `), and individual characters (`""`).
* **Chunk Size (`1000` chars):** Bounds each segment to a focused semantic concept optimized for embedding models and vector comparison.
* **Chunk Overlap (`200` chars):** Creates a sliding context window across chunk boundaries, ensuring that ideas, technical terms, or relationships split across two segments are not lost during retrieval.
* **Metadata Persistence:** LangChain preserves document metadata (such as original source file and page numbers) across every chunk, enabling precise traceability and metadata filtering later in the pipeline.

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configure the hierarchical text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

# Split the loaded PDF documents
splits = text_splitter.split_documents(docs)

# Diagnostics & Inspection
print(f"Total source pages loaded : {len(docs)}")
print(f"Total text chunks created : {len(splits)}")
print(f"Average chunk character length: {sum(len(s.page_content) for s in splits) // len(splits)}")
print("=" * 80)
print("Sample Chunk [Index 0]:\n")
print(splits[0].page_content)
print("\nAssociated Metadata:")
print(splits[0].metadata)
print("=" * 80)

Total source pages loaded : 12
Total text chunks created : 47
Average chunk character length: 880
Sample Chunk [Index 0]:

Journal of Physics:
Conference Series
       
PAPER • OPEN ACCESS
Evaluation of the Robustness of Machine Learning
Surrogate Models for Wind Turbine Fatigue Load
Estimation using SCADA data
To cite this article: Samy Zouhri et al 2026 J. Phys.: Conf. Ser. 3224 062038
 
View the article online for updates and enhancements.
You may also like
Real-Time Two-Step Alignment
Framework for Nacelle-Mounted LiDAR
Systems in Wind Farms
Konstantinos Vratsinis, Jonathan Sterckx,
Pieter-Jan Daems et al.
-
Enhancing Wind Turbine Reliability via
Factor Analysis of Vibration Trends
Taiane Pereira Reis, Marina Geremia,
Glauco Cardozo et al.
-
Wind Turbine Yaw Misalignment Detection
Based on Simulated Data
Honovan Paz Rocha, Edwan Anderson
Ariza Echeverri, Luiz David Ricarte de
Souza Custodio et al.
-
This content was downloaded from IP address 161.123.167.126 on 28/05/2026 at 20:53


### Step 3: Embedding and Vector Storage

**Overview & Strategy:**
To enable semantic search, our text chunks must be translated into a mathematical format. We use an embedding model to convert each chunk into a high-dimensional vector. These vectors are then stored in a specialized vector database (Chroma) that optimizes for fast similarity searches.

**Key Mechanics:**
* **Dense Vector Embeddings:** We use a HuggingFace sentence-transformer model (`all-MiniLM-L6-v2`) to embed the text locally and for free. This model captures the semantic context of the text.
* **ChromaDB:** A lightweight vector database that ingests our documents and their corresponding embeddings.
* **Approximate Nearest Neighbor (ANN):** Chroma utilizes ANN algorithms (like HNSW) to rapidly retrieve chunks whose vectors are closest to the embedded user query in the vector space.
* **Persistence:** We save the vector database to the local disk (`data/chroma_db/`) to prevent needing to re-compute embeddings every time the notebook is run.

In [21]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import os

# 1. Initialize the locally hosted, open-source embedding model
print("Loading HuggingFace embedding model...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Define the local directory to store the database files
persist_directory = 'data/chroma_db/'

# Clean up existing database if running this cell multiple times (optional)
if os.path.exists(persist_directory):
    import shutil
    shutil.rmtree(persist_directory)

# 3. Create the vector database and embed the documents
print(f"Building Chroma vector store at '{persist_directory}'...")
vectordb = Chroma.from_documents(
    documents=splits,
    embedding=embedding_model,
    persist_directory=persist_directory
)

# Diagnostics & Inspection
print("=" * 80)
print(f"Successfully stored {vectordb._collection.count()} chunks in the database.")
print("=" * 80)

Loading HuggingFace embedding model...

Building Chroma vector store at 'data/chroma_db/'...
Successfully stored 47 chunks in the database.


### Step 4: Retrieval & Similarity Search

**Overview & Strategy:**
We configure our Chroma database as a retrieval engine to fetch the most relevant context for a given query. 

**Key Mechanics:**
* **Maximum Marginal Relevance (MMR):** Instead of standard similarity search, we use MMR to prevent retrieving redundant chunks. MMR balances semantic relevance with diversity, ensuring the LLM gets a comprehensive view of the topic rather than three identical sentences.
* **Retriever Interface:** We wrap the vector database in LangChain's `.as_retriever()` method, standardizing it for the final LCEL (LangChain Expression Language) pipeline.

In [22]:
# Define a test question relevant to your machine learning robustness paper
question = "How does the paper evaluate the robustness of machine learning models?"

# Option A: Direct similarity search (Without MMR) - For demonstration
print("--- Standard Similarity Search ---")
basic_docs = vectordb.similarity_search(question, k=2)
for i, doc in enumerate(basic_docs):
    print(f"Result {i+1}:\n{doc.page_content[:200]}...\n")

# Option B: Advanced Retrieval using MMR - This is what we will use for the final pipeline
print("\n--- MMR Retrieval (Relevance + Diversity) ---")
retriever = vectordb.as_retriever(
    search_type="mmr", 
    search_kwargs={"k": 3, "fetch_k": 10}
)

mmr_docs = retriever.invoke(question)
for i, doc in enumerate(mmr_docs):
    # Notice the metadata inclusion (useful for citing sources in the final answer)
    print(f"Result {i+1} (Page {doc.metadata.get('page', 'N/A')}):")
    print(f"{doc.page_content}\n")
    print("-" * 80)

--- Standard Similarity Search ---
Result 1:
plemented during model training. First, a K-fold cross-validation procedure is used during both 
the training and hyperparameter optimization phases to evaluate the model performance on un-
seen subse...

Result 2:
aero-servo-elastic simulations generated from framework such as FAST [3] or BHAWC [4]. While 
these simulation platforms provide valuable environments for assessing model behavior, they do 
not fully ...


--- MMR Retrieval (Relevance + Diversity) ---
Result 1 (Page 5):
plemented during model training. First, a K-fold cross-validation procedure is used during both 
the training and hyperparameter optimization phases to evaluate the model performance on un-
seen subsets of the data. This approach allows the model to be trained and validated across mul-
tiple data partitions, providing a more reliable estimate of its predictive capability on unseen data. 
In addition, the complexity of the XGBoost model is controlled through hyperpa

### Step 5: Augmentation and Generation (LCEL Pipeline)

**Overview & Strategy:**
We connect the retrieved context to a Large Language Model to generate a final answer. We use LangChain Expression Language (LCEL) to build a streamlined, production-ready pipeline.

**Key Mechanics:**
* **PromptTemplate:** A strict set of instructions forcing the LLM to rely solely on the provided context, minimizing hallucinations.
* **RunnablePassthrough:** An LCEL component that cleanly routes the user's query to both the retriever and the final prompt.
* **StrOutputParser:** Strips away the LLM's metadata, returning a clean string for the user.

In [25]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)
models = response.json()

print("Available Models on your tier:")
for model in models.get('data', []):
    print(f"- {model['id']}")

Available Models on your tier:
- canopylabs/orpheus-arabic-saudi
- qwen/qwen3.8-27b
- openai/gpt-oss-20b
- openai/gpt-oss-120b
- meta-llama/llama-prompt-guard-2-86m
- whisper-large-v3
- allam-2-7b
- meta-llama/llama-prompt-guard-2-22m
- canopylabs/orpheus-v1-english
- whisper-large-v3-turbo
- openai/gpt-oss-safeguard-20b


In [26]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

# Load environment variables (Make sure GROQ_API_KEY is in your .env file)
load_dotenv()

# # 1. Initialize the free LLM (Groq running Llama 3)
# llm = ChatGroq(model_name="llama3-8b-8192", temperature=0)

# Updated initialization using a confirmed available text generation model
llm = ChatGroq(model_name="qwen/qwen3.8-27b", temperature=0)

# 2. Define the Prompt Template (including the specific instructions from the assignment)
template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Always say "thanks for asking!" at the end of the answer.

{context}

Question: {question}
Helpful Answer:"""

QA_PROMPT = PromptTemplate.from_template(template)

def format_docs(docs):
    """Helper function to combine retrieved document chunks into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

# 3. Build the LCEL Chain
rag_chain = (
    # RunnableParallel implicitly created by the dictionary
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | QA_PROMPT         # Augmentation
    | llm               # Generation
    | StrOutputParser() # Output Formatting
)

# 4. Invoke the pipeline with a test query
query = "How does the paper evaluate the robustness of machine learning models?"
print(f"Question: {query}\n")
print("-" * 80)

# Generate the response
response = rag_chain.invoke(query)
print(response)
print("-" * 80)

Question: How does the paper evaluate the robustness of machine learning models?

--------------------------------------------------------------------------------
Based on the provided context, the paper evaluates the robustness of the machine learning models through the following methods:

1.  **K-fold Cross-Validation:** This procedure is used during both the training and hyperparameter optimization phases. It allows the model to be trained and validated across multiple data partitions, providing a more reliable estimate of predictive capability on unseen data.
2.  **Hyperparameter Optimization:** The complexity of the XGBoost model is controlled by tuning parameters such as maximum tree depth, learning rate, and minimum child weight. These are optimized using a Bayesian optimization approach to maximize predictive performance while limiting excessive model complexity.
3.  **Validation Framework:** The text notes that validation is a critical step to assess reliability for industrial

In [ ]:
# !pip install langchain-groq

### Test: 

In [29]:
import textwrap
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

def generate_rag_answer(query: str, vector_store, llm, line_width: int = 80) -> str:
    """
    Takes a user query, retrieves diverse context from the vector database using MMR, 
    and generates a grounded answer. Wraps the output text to prevent horizontal 
    scrolling in Jupyter Notebooks.
    """
    # 1. Configure the retriever for Maximum Marginal Relevance (MMR)
    retriever = vector_store.as_retriever(
        search_type="mmr", 
        search_kwargs={"k": 3, "fetch_k": 10}
    )
    
    # 2. Define the exact formatting instructions
    template = """Use the following pieces of context to answer the question at the end.
    If you don't know the answer, just say that you don't know, don't try to make up an answer.
    Always say "thanks for asking!" at the end of the answer.

    {context}

    Question: {question}
    Helpful Answer:"""
    
    qa_prompt = PromptTemplate.from_template(template)
    
    # 3. Helper function to stitch chunks into a single text block
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)
        
    # 4. Build the LCEL Chain
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | qa_prompt
        | llm
        | StrOutputParser()
    )
    
    # 5. Execute the pipeline to get the raw string
    raw_answer = rag_chain.invoke(query)
    
    # 6. Format the text for readability (preventing horizontal scrolling)
    paragraphs = raw_answer.split('\n')
    wrapped_paragraphs = [
        textwrap.fill(p, width=line_width) if p.strip() else p 
        for p in paragraphs
    ]
    
    return '\n'.join(wrapped_paragraphs)

In [30]:
# Cell: Execute 5 Specific Questions on the Research Paper
import time

# Define the 5 analytical questions for the wind turbine surrogate load model paper
evaluation_questions = [
    "What specific machine learning algorithms or architectures were used to develop the surrogate models for load estimation?",
    "What environmental and operational variables were selected as input features for the wind turbine models?",
    "How does the paper evaluate the robustness of the surrogate models compared to traditional aeroelastic simulations?",
    "How are the load estimates generated by the machine learning model specifically used in the wind turbine siting process?",
    "What limitations or specific operating conditions does the paper identify where the surrogate model struggles to accurately predict loads?"
]

# Ensure you have already run the cells that initialize `vectordb` and `llm` before running this cell.
print("Starting RAG evaluation...\n" + "="*80)

# Loop through each question, generate the answer using the LCEL pipeline, and print the results
for i, question in enumerate(evaluation_questions, 1):
    print(f"\nQuestion {i}: {question}\n")
    
    try:
        # Call the modular RAG function we defined earlier
        answer = generate_rag_answer(
            query=question, 
            vector_store=vectordb, 
            llm=llm
        )
        print("Generated Answer:")
        print(answer)
    except Exception as e:
        print(f"An error occurred while generating the answer: {e}")
    
    print("\n" + "-"*80)
    
    # Adding a brief pause to respect free-tier API rate limits (Groq has rate limits per minute)
    time.sleep(2)

print("\nEvaluation complete.")

Starting RAG evaluation...

Question 1: What specific machine learning algorithms or architectures were used to develop the surrogate models for load estimation?

Generated Answer:
Based on the provided context, the specific machine learning algorithms or
architectures used to develop the surrogate models are not mentioned. The text
refers to "machine-learning surrogate models" and "machine-learning models"
generally, but does not specify the particular type (e.g., neural networks,
random forests, etc.).

thanks for asking!

--------------------------------------------------------------------------------

Question 2: What environmental and operational variables were selected as input features for the wind turbine models?

Generated Answer:
Based on the provided context, the text explicitly mentions a **variable
indicating Power Boost activation** as an input feature added to the machine-
learning model's input space to help the algorithm adapt to heterogeneous
control settings. Additio

### RAG Pipeline Architecture

The following flowchart outlines the end-to-end Retrieval-Augmented Generation pipeline implemented in this notebook, from raw PDF ingestion to the final generated response.

```mermaid
graph TD
    subgraph Data Ingestion Phase
        A[📄 Wind Turbine Research Paper PDF] -->|PyPDFLoader| B(Extracted Documents)
        B -->|RecursiveCharacterTextSplitter| C(Text Chunks)
        C -->|HuggingFace: all-MiniLM-L6-v2| D[(Chroma Vector DB)]
    end

    subgraph Retrieval & Generation Phase
        E[👤 User Query] -->|MMR Similarity Search| D
        D -->|Top 3 Diverse Chunks| F[Retrieved Context]
        E --> G{Prompt Template}
        F --> G
        G -->|LCEL Pipeline| H((ChatGroq: Qwen 27B))
        H -->|StrOutputParser + TextWrap| I[💬 Grounded & Formatted Answer]
    end
    
    classDef database fill:#f9f0ff,stroke:#d4b3ff,stroke-width:2px;
    class D database;